# **Module 4 — AI HEALTH CHATBOT**
## MedAI Nexus | Gemini 1.5 Flash Conversational Assistant
---
- **Model:** Gemini 1.5 Flash (Free API)
- **Task:** Answer health questions grounded in the user's own module results
- **Goal:** Accept user query → Check emergency → Inject health context → Build system prompt → Initialize Gemini chat (with memory) → Generate personalized response → Clean & control output → Log conversation → Display response

**Objective:**
To develop a context-aware AI health chatbot that delivers personalized, safe, and easy-to-understand healthcare guidance by leveraging user-specific data from previous modules.

## 1. Install & Import
---

In [1]:
!pip install google-generativeai -q

In [2]:
import os
import json
import time
import uuid
import warnings
warnings.filterwarnings('ignore')

import google.generativeai as genai
from google.colab import userdata

print("Libraries loaded.")

Libraries loaded.


## 2. Configure Gemini API
---

In [3]:

GEMINI_API_KEY = userdata.get('Gemini_API_Key')
genai.configure(api_key=GEMINI_API_KEY)

print(" Gemini API configured.")
print("Model: gemini-1.5-flash")

 Gemini API configured.
Model: gemini-1.5-flash


##3. Load session_state results
---

In [4]:
# TEMP: Replace this with Streamlit session_state later

skin_result = {
    "disease": "Acne",
    "confidence": "87%"
}

risk_result = {
    "risk_level": "High",
    "probability": "64%"
}

report_result = {
    "summary": "High glucose levels detected. Possible diabetes risk."
}

##4. Build structured context
---

In [5]:
context = f"""
USER HEALTH DATA:

1. Skin Disease Result:
{skin_result}

2. Diabetes Risk:
{risk_result}

3. Medical Report Summary:
{report_result}
"""

##5. System Prompt
---
The system prompt is the permanent instruction we give Gemini before any conversation starts.
It :
- Defines the chatbot’s role as a healthcare assistant
- Injects user’s health data for personalized responses
- Ensures safe responses (no diagnosis, only guidance)
- Controls tone (simple, clear, user-friendly)
- Adds rules for handling serious conditions (advise doctor)


In [6]:
def build_system_prompt(health_context: dict) -> str:
    """
    Build a dynamic system prompt using the user's health data.

    Args:
        health_context : dict — results from Modules 1, 2, 3
                         Keys: 'skin_result', 'risk_result', 'report_result'

    Returns:
        str — complete system prompt for chatbot session
    """

    # ── Safe extraction of each module's result (handles missing or None values) ──
    skin   = health_context.get('skin_result') or 'Not assessed yet'
    risk   = health_context.get('risk_result') or 'Not assessed yet'
    report = health_context.get('report_result') or 'Not assessed yet'

    # ── Build system prompt ──
    prompt = f"""You are MedBot, a compassionate and knowledgeable AI health assistant.
You are part of MedAI Nexus, an AI-powered healthcare assistant system.

ABOUT THIS USER (from previous modules):
- Skin disease result: {skin}
- Diabetes risk assessment: {risk}
- Lab report findings: {report}

YOUR RULES — follow strictly:

1. PERSONALISE every response using the user’s data above.
   Avoid generic advice — refer to their specific results.

2. NEVER diagnose any condition.
   Do NOT say 'You have diabetes' or 'You have eczema'.
   Instead say 'Your assessment suggests...' or 'Your report indicates...'

3. ALWAYS recommend consulting a qualified doctor for medical decisions.

4. EMERGENCY HANDLING:
   If the user mentions symptoms like chest pain, breathing difficulty,
   suicidal thoughts, collapse, or severe bleeding —
   respond with:
   "⚠️ Please contact emergency services (112 / 108 in India) immediately."
   Do not continue normal conversation.

5. STAY within health-related topics only.
   Politely redirect non-health queries.

6. TONE:
   - Warm, supportive, and simple English
   - Avoid medical jargon
   - If used, explain it in brackets

7. RESPONSE LENGTH:
   Keep answers concise (3–5 sentences) unless more detail is needed.

8. If the question is unclear or vague, ask a follow-up clarification.

9. Use bullet points when giving advice or recommendations.

NOTE:
Emergency detection is ALSO handled in application logic.
Do not rely only on this prompt for safety.
"""

    return prompt


print("System prompt builder defined.")
print("It will personalise every conversation using the user's module results.")

System prompt builder defined.
It will personalise every conversation using the user's module results.


## 6. The Chatbot Class
---
Gemini (like all LLMs) has NO memory between API calls.
Each call is completely independent.
- Here a list called `conversation_history` powered by Gemini 1.5 Flash is maintained
- Maintains conversation history to simulate memory.

In [7]:
class MedAIChatbot:

    # Initialise the chatbot
    #--------------------------
    def __init__(self, health_context: dict):
        """
        Initialise the chatbot with the user's health results.

        Args:
            health_context : dict — {'skin_result': ..., 'risk_result': ..., 'report_result': ...}
        """

        # ── Store the user's health context ──────────────────────────────────
        self.health_context = health_context

        # ── Build the Gemini model ────────────────────────────────────────────
        from google.generativeai.types import GenerationConfig  # add at top of file

        self.model = genai.GenerativeModel(
            model_name="gemini-1.5-flash",
            system_instruction=build_system_prompt(health_context),
            generation_config=GenerationConfig(
            temperature=0.3
            )
        )

        # ── Start a chat session ──────────────────────────────────────────────
        self.chat_session = self.model.start_chat(history=[])

        # ── Emergency keywords ────────────────────────────────────────────────
        self.emergency_keywords = [
            'chest pain', 'cannot breathe', "can't breathe", 'heart attack',
            'suicidal', 'want to die', 'collapse', 'unconscious', 'stroke',
            'seizure', 'overdose', 'bleeding heavily'
        ]

        # ── Conversation log ──────────────────────────────────────────────────
        self.conversation_log = []
        self.session_id = str(uuid.uuid4())

        print(" MedBot initialised.")
        print(f"     Health context loaded: {list(health_context.keys())}")




    # Emergency Detection Layer
    #-----------------------------
    def _is_emergency(self, message: str) -> bool:
        """
        Check if user message contains emergency keywords.

        Args:
            message : str — the user's message

        Returns:
            bool — True if emergency keywords found
        """
        # Convert to lowercase for case-insensitive matching
        msg_lower = message.lower()
        return any(keyword in msg_lower for keyword in self.emergency_keywords)                  # No emergency keywords found


    def _get_emergency_response(self) -> str:
        """Return a fixed emergency message — no API call needed."""
        return (
            "URGENT: Based on what you described, please CALL EMERGENCY SERVICES IMMEDIATELY.\n"
            "In India: Call 112 (national emergency) or 108 (ambulance).\n"
            "Please stop reading this and call for help now. "
            "This is not something to manage at home."
        )


    def _clean_response(self, text: str) -> str:
        """Clean and format model response."""
        return text.strip()




    # Send message
    #----------------------

    def send_message(self, user_message: str) -> str:
        """
        Send a message and get a response.

        Args:
            user_message : str — what the user typed

        Returns:
            str — MedBot's response
        """

        # ── Step 1: Emergency check ────────────────────────────────────────────
        # This runs BEFORE Gemini — if emergency, return fixed message
        from datetime import datetime
        if self._is_emergency(user_message):
            response_text = self._get_emergency_response()
            self.conversation_log.append({
                'role': 'user',
                'content': user_message,
                'time': datetime.now().strftime("%H:%M:%S")
             })
            self.conversation_log.append({
                 'role': 'assistant',
                 'content': response_text,
                 'time': datetime.now().strftime("%H:%M:%S")
             })

            return response_text

        # ── Step 2: Send to Gemini ─────────────────────────────────────────────
        try:
            response = self.chat_session.send_message(user_message)
            # Step 1: Clean response
            response_text = self._clean_response(response.text)
            # Step 2: Limit response length (ADD THIS HERE)
            if len(response_text) > 800:
               response_text = response_text[:800] + "..."
        except Exception as e:
            # Handle API errors gracefully — don't crash the app
            response_text = (
                f"I'm sorry, I had trouble responding right now. "
                f"Please try again in a moment. (Error: {str(e)[:80]})"
            )

        # ── Step 3: Log the exchange ──────────────────────────────────────────
        self.conversation_log.append({'role': 'user',      'content': user_message})
        self.conversation_log.append({'role': 'assistant', 'content': response_text})

        return response_text


    # Summary
    #-----------
    def get_conversation_summary(self) -> dict:
        """Return a summary of the conversation for saving or displaying."""
        return {
            'total_turns'    : len(self.conversation_log) // 2,
            'health_context' : self.health_context,
            'messages'       : self.conversation_log,
        }



    # Fresh conversation
    #---------------------
    def reset(self):
        """Start a fresh conversation (keeps the same health context)."""
        self.chat_session    = self.model.start_chat(history=[])
        self.conversation_log = []
        print("Conversation reset. Health context preserved.")


print("MedAIChatbot class defined.")
print("Key design: emergency check runs BEFORE Gemini (fast, no API cost).")

MedAIChatbot class defined.
Key design: emergency check runs BEFORE Gemini (fast, no API cost).


##7. Test the Chatbot
---
Test with a sample patient who has results from all three modules.


In [8]:
# Simulated health context — in Streamlit this comes from session_state
sample_health_context = {
    'skin_result'  : 'Eczema Photos (78.4% confidence)',
    'risk_result'  : 'Moderate risk (48.2% probability)',
    'report_result': 'HbA1c: 6.1% (slightly elevated). Haemoglobin: Normal. Cholesterol: Borderline high.'
}

# Create the chatbot
bot = MedAIChatbot(health_context=sample_health_context)
print()

 MedBot initialised.
     Health context loaded: ['skin_result', 'risk_result', 'report_result']



In [9]:
# --- Test conversation ---

questions = [
    "Hello! What do my results mean overall?",
    "What diet should I follow based on my results?",
    "My skin is itchy. Is that related to my eczema result?",
    "Should I be worried about my HbA1c?",
]

for q in questions:
    print(f"\nUSER: {q}")
    response = bot.send_message(q)
    print(f"MEDBOT: {response}")
    print("-" * 60)
    time.sleep(1)  # Small pause to avoid rate limits


USER: Hello! What do my results mean overall?


MEDBOT: I'm sorry, I had trouble responding right now. Please try again in a moment. (Error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flas)
------------------------------------------------------------

USER: What diet should I follow based on my results?


MEDBOT: I'm sorry, I had trouble responding right now. Please try again in a moment. (Error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flas)
------------------------------------------------------------

USER: My skin is itchy. Is that related to my eczema result?


MEDBOT: I'm sorry, I had trouble responding right now. Please try again in a moment. (Error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flas)
------------------------------------------------------------

USER: Should I be worried about my HbA1c?


MEDBOT: I'm sorry, I had trouble responding right now. Please try again in a moment. (Error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flas)
------------------------------------------------------------


In [10]:
# --- Test emergency detection ---

print("Testing emergency detection...")
print("(This bypasses Gemini — no API call is made)\n")

emergency_msg = "I have severe chest pain and can't breathe"
print(f"USER: {emergency_msg}")
response = bot.send_message(emergency_msg)
print(f"MEDBOT: {response}")

Testing emergency detection...
(This bypasses Gemini — no API call is made)

USER: I have severe chest pain and can't breathe
MEDBOT: URGENT: Based on what you described, please CALL EMERGENCY SERVICES IMMEDIATELY.
In India: Call 112 (national emergency) or 108 (ambulance).
Please stop reading this and call for help now. This is not something to manage at home.


## 8. Save Conversation
---

In [11]:
import shutil, pickle

# Save conversation log
summary = bot.get_conversation_summary()
with open('chatbot_conversation.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Copy to Drive
drive_dir = '/content/drive/MyDrive/medai_chatbot/'
os.makedirs(drive_dir, exist_ok=True)
shutil.copy('chatbot_conversation.json', drive_dir)

print(f"Conversation saved: {summary['total_turns']} turns")
print(f"Saved to Drive: {drive_dir}")

Conversation saved: 5 turns
Saved to Drive: /content/drive/MyDrive/medai_chatbot/


## 9. Inference Function
---

In [12]:
def get_or_create_chatbot(health_context: dict, session_state) -> MedAIChatbot:
    """
    Get existing chatbot from session or create a new one.

    Args:
        health_context : dict       — from st.session_state (module results)
        session_state  : object     — st.session_state

    Returns:
        MedAIChatbot — ready to use
    """
    if 'chatbot' not in session_state:
        # First time on this page — create a new chatbot
        session_state.chatbot = MedAIChatbot(health_context)
        session_state.messages = []     # list of {role, content} dicts for display

    return session_state.chatbot


def chat_response(user_message: str, bot: MedAIChatbot) -> str:
    """
    Get response for one user message.

    Args:
        user_message : str           — what the user typed
        bot          : MedAIChatbot  — the chatbot instance

    Returns:
        str — MedBot's response text
    """
    return bot.send_message(user_message)